# Multilingual Political & Fake News Classification — Example Notebook

This notebook demonstrates how to call the scripts in this repository for:
1. **Installing dependencies**
2. **Data preparation**
3. **Supervised fine-tuning** (encoder & decoder models)
4. **Zero-shot / Chain-of-Thought evaluation**
5. **Few-shot in-context learning evaluation**
6. **Loading and analyzing results**

> **Note:** GPU-intensive steps (training, LLM inference) require a CUDA-capable GPU.
> Adjust `--model_name` and dataset parameters to match your setup.

## 0. Setup — Install Dependencies

In [ ]:
# Install all required packages
!pip install -r requirements.txt

In [ ]:
import os
import json
import subprocess

# Set your HuggingFace token if using gated models (Llama, Mistral)
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# Set wandb to offline mode if you don't want to log to the cloud
os.environ["WANDB_MODE"] = "offline"

## 1. Data Preparation

Before training or evaluation, you must prepare the dataset.

### Supported datasets and their task labels:
| Dataset Name | `--dataset_name` | `--task_labels` | Supported Languages |
|---|---|---|---|
| Hyperpartisan News Headlines | `hyperpartisan_news_headlines` | `hp` | `en` |
| SemEval 2019 | `semeval_2019` | `hp` | `en` |
| AllSides | `all_sides` | `pl` | `en` |
| CLEF 3A | `clef_3a` | `pl` | `en` |
| CLEF 1C | `clef_1c` | `ht` | `en`, `bg`, `ar` |
| FakeNewsNet | `fake_news_net` | `fn` | `en` |
| Fake.br Corpus | `fake_br_corpus` | `fn` | `pt` |
| Fake News Corpus Spanish | `fake_news_corpus_spanish` | `fn` | `es` |

Place your raw dataset files under `./data/<dataset_name>/` before running.

In [ ]:
# Example: Prepare the Hyperpartisan News Headlines dataset
!python prepare_data.py \
    --dataset_name hyperpartisan_news_headlines \
    --language en

In [ ]:
# Example: Prepare the CLEF 1C dataset in Bulgarian
!python prepare_data.py \
    --dataset_name clef_1c \
    --language bg

In [ ]:
# Verify the processed data was created
!ls -la ./processed_data/

In [ ]:
# Peek at the processed data
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "./processed_data/train.json",
        "test": "./processed_data/test.json",
    },
)
print(dataset)
print("\nSample train example:")
print(dataset["train"][0])
print(f"\nNumber of labels: {len(dataset['train'].unique('label'))}")

## 2. Supervised Fine-Tuning — Encoder Models

Encoder models (RoBERTa, mDeBERTa, ModernBERT, etc.) are fine-tuned with LoRA adapters.

### Available encoder models:
- `FacebookAI/roberta-base`
- `FacebookAI/roberta-large`
- `FacebookAI/xlm-roberta-base`
- `microsoft/mdeberta-v3-base`
- `answerdotai/ModernBERT-base`
- `launch/POLITICS`

In [ ]:
# Fine-tune RoBERTa-base on the prepared dataset
!python train_encoder_seq_cls.py \
    --model_name FacebookAI/roberta-base \
    --dataset_name hyperpartisan_news_headlines \
    --epochs 3 \
    --runs 1 \
    --batch_size 8 \
    --lr 1e-4

In [ ]:
# Fine-tune with 4-bit quantization (requires bitsandbytes + CUDA GPU)
# Note: When --use_quantization is enabled, the script automatically uses
# device_map="auto" (required by bitsandbytes for proper FP4 state initialization).
!python train_encoder_seq_cls.py \
    --model_name FacebookAI/roberta-large \
    --dataset_name hyperpartisan_news_headlines \
    --epochs 3 \
    --runs 5 \
    --batch_size 8 \
    --use_quantization

## 3. Supervised Fine-Tuning — Decoder Models

Decoder models (Llama, Mistral, Qwen) are also fine-tuned with LoRA.

### Available decoder models:
- `meta-llama/Meta-Llama-3.1-8B` (access request required)
- `meta-llama/Meta-Llama-3.1-8B-Instruct` (access request required)
- `mistralai/Mistral-Nemo-Instruct-2407` (access request required)
- `Qwen/Qwen2.5-7B-Instruct`

In [ ]:
# Fine-tune Qwen2.5-7B-Instruct (no access request needed)
!python train_decoder_seq_cls.py \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --dataset_name hyperpartisan_news_headlines \
    --epochs 3 \
    --runs 1 \
    --batch_size 4 \
    --use_quantization

## 4. Zero-Shot / Chain-of-Thought Evaluation

Evaluate LLMs using various prompting strategies:
- `zero_shot_generic` — Simple zero-shot classification
- `zero_shot_specific` — Zero-shot with task-specific definition
- `codebook` — Uses a linguistic feature codebook
- `cot` — Chain-of-Thought step-by-step reasoning

Label types: `string` (text labels) or `int` (integer labels)

In [ ]:
# Zero-shot generic evaluation on Hyperpartisan detection
!python eval_zero_cot.py \
    --model_name meta-llama/Llama-3.1-8B-Instruct \
    --dataset_name hyperpartisan_news_headlines \
    --configuration zero_shot_generic \
    --task_labels hp \
    --label_type string \
    --language en \
    --verbose

In [ ]:
# Chain-of-Thought evaluation on Fake News detection
!python eval_zero_cot.py \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --dataset_name fake_news_net \
    --configuration cot \
    --task_labels fn \
    --label_type string \
    --language en \
    --verbose

In [ ]:
# Codebook evaluation with integer labels
!python eval_zero_cot.py \
    --model_name meta-llama/Llama-3.1-8B-Instruct \
    --dataset_name hyperpartisan_news_headlines \
    --configuration codebook \
    --task_labels hp \
    --label_type int \
    --language en

## 5. Few-Shot In-Context Learning Evaluation

Two strategies for selecting few-shot examples:
- `fs_dpp` — Determinantal Point Process (DPP) selected examples
- `fs_random` — Randomly sampled examples

The script automatically sweeps over different numbers of shots (1-10).

In [ ]:
# Few-shot with random examples on Hyperpartisan detection
!python eval_few_shot.py \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --dataset_name hyperpartisan_news_headlines \
    --configuration fs_random \
    --task_labels hp \
    --label_type string \
    --language en \
    --runs 3 \
    --verbose

In [ ]:
# Few-shot with DPP-selected examples
# NOTE: Requires pre-computed DPP example files in ./data/<dataset_name>/
!python eval_few_shot.py \
    --model_name meta-llama/Llama-3.1-8B-Instruct \
    --dataset_name hyperpartisan_news_headlines \
    --configuration fs_dpp \
    --task_labels hp \
    --label_type string \
    --language en \
    --runs 5

In [ ]:
# Few-shot on specific shot range (e.g., only 2-shot and 4-shot for binary tasks)
!python eval_few_shot.py \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --dataset_name hyperpartisan_news_headlines \
    --configuration fs_dpp \
    --task_labels hp \
    --label_type string \
    --language en \
    --start 2 \
    --end 4 \
    --runs 5

## 6. Loading & Analyzing Results

After running experiments, the scripts save JSON result files.
Here's how to load and visualize them.

In [ ]:
import json
import pandas as pd

# ---- Load zero-shot / CoT results ----
results_path = "results.json"  # Generated by eval_zero_cot.py

try:
    with open(results_path, "r") as f:
        results = json.load(f)
    print("=== Zero-Shot / CoT Results ===")
    print(json.dumps(results, indent=4))
except FileNotFoundError:
    print(f"No results file found at {results_path}.")
    print("Run an evaluation script first to generate results.")

In [ ]:
# ---- Load model outputs (predictions vs ground truth) ----
outputs_path = "model_outputs.json"  # Generated by eval_zero_cot.py

try:
    with open(outputs_path, "r") as f:
        model_outputs = json.load(f)
    print(f"Total predictions: {len(model_outputs['model_predictions'])}")
    print(f"Total ground truth: {len(model_outputs['ground_truth'])}")
    
    # Show first 10 predictions vs ground truth
    for i in range(min(10, len(model_outputs['model_predictions']))):
        pred = model_outputs['model_predictions'][i]
        gt = model_outputs['ground_truth'][i]
        match = '✓' if pred == gt else '✗'
        print(f"  [{match}] Pred: {pred}  |  GT: {gt}")
except FileNotFoundError:
    print(f"No model outputs found at {outputs_path}.")

In [ ]:
# ---- Load few-shot averaged results ----
avg_results_path = "avg_results.json"  # Generated by eval_few_shot.py

try:
    with open(avg_results_path, "r") as f:
        avg_results = json.load(f)
    print("=== Few-Shot Averaged Results ===")
    print(json.dumps(avg_results, indent=4))
except FileNotFoundError:
    print(f"No averaged results found at {avg_results_path}.")

In [ ]:
# ---- Load unparseable outputs (failure analysis) ----
unparseable_path = "unparseable_outputs.json"  # Generated when parsing fails

try:
    with open(unparseable_path, "r") as f:
        unparseable = json.load(f)
    print(f"=== Unparseable Outputs: {len(unparseable)} total ===")
    for entry in unparseable[:5]:  # Show first 5
        print(f"  Index: {entry['index']}")
        print(f"  Ground Truth: {entry['ground_truth']}")
        print(f"  Raw Output: {entry['output'][:200]}...")
        print()
except FileNotFoundError:
    print("No unparseable outputs file found (good — all outputs were parsed).")

## 7. Using the Utility Functions Directly

You can import and use the utility functions from `utils.py` for custom analysis.

In [ ]:
from utils import compute_metrics, compute_average_metrics, get_dataset_length_stats

# Example: Compute metrics from predictions and labels
predictions = [0, 1, 1, 0, 1, 0, 0, 1, 1, 0]
labels =      [0, 1, 0, 0, 1, 1, 0, 1, 1, 0]

metrics = compute_metrics(predictions, labels)
print("=== Computed Metrics ===")
print(json.dumps(metrics, indent=4))

In [ ]:
# Example: Average metrics across multiple runs
run_results = [
    {"accuracy": 0.85, "precision": 0.83, "recall": 0.82, "f1_score": 0.82},
    {"accuracy": 0.87, "precision": 0.86, "recall": 0.84, "f1_score": 0.85},
    {"accuracy": 0.84, "precision": 0.82, "recall": 0.81, "f1_score": 0.81},
]

avg = compute_average_metrics(run_results)
print("=== Averaged Metrics ===")
for metric, values in avg.items():
    print(f"  {metric}: {values['score']:.4f} ± {values['stddev']:.4f} "
          f"(min={values['min']:.4f}, max={values['max']:.4f})")

In [ ]:
# Example: Get token length statistics for a dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

try:
    dataset = load_dataset(
        "json",
        data_files={
            "train": "./processed_data/train.json",
            "test": "./processed_data/test.json",
        },
    )
    stats = get_dataset_length_stats(tokenizer, dataset)
    print("=== Token Length Statistics ===")
    print(json.dumps(stats, indent=4))
except FileNotFoundError:
    print("Run prepare_data.py first to generate processed data.")

## 8. Visualization Example — Comparing Results

Below is a template for plotting results from multiple experiments.

In [ ]:
# pip install matplotlib  # Uncomment if not installed
import matplotlib.pyplot as plt

# Example data — replace with your actual results
models = ["RoBERTa-base", "RoBERTa-large", "mDeBERTa", "Llama-3.1-8B", "Qwen2.5-7B"]
f1_scores = [0.82, 0.85, 0.83, 0.78, 0.80]
accuracy = [0.84, 0.87, 0.85, 0.79, 0.81]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

x = range(len(models))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], f1_scores, width, label="F1 Score", color="steelblue")
bars2 = ax.bar([i + width/2 for i in x], accuracy, width, label="Accuracy", color="coral")

ax.set_ylabel("Score")
ax.set_title("Model Comparison — Hyperpartisan Detection")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Example: Plot few-shot performance by number of shots
shots = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
llama_f1 =  [0.55, 0.60, 0.63, 0.66, 0.68, 0.70, 0.71, 0.72, 0.73, 0.74]
qwen_f1 =   [0.58, 0.62, 0.65, 0.67, 0.69, 0.70, 0.72, 0.73, 0.74, 0.75]

plt.figure(figsize=(10, 5))
plt.plot(shots, llama_f1, marker="o", label="Llama-3.1-8B-Instruct", linewidth=2)
plt.plot(shots, qwen_f1, marker="s", label="Qwen2.5-7B-Instruct", linewidth=2)

plt.xlabel("Number of Shots")
plt.ylabel("F1 Score")
plt.title("Few-Shot Performance Scaling")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(shots)
plt.tight_layout()
plt.show()

## 9. Multilingual Example — CLEF 1C (Harmful Tweet Detection)

This dataset supports English, Bulgarian, and Arabic.
You can run fine-tuning and evaluation in any supported language.

In [ ]:
# Step 1: Prepare CLEF 1C dataset for Bulgarian
!python prepare_data.py --dataset_name clef_1c --language bg

# Step 2: Fine-tune a multilingual encoder model
!python train_encoder_seq_cls.py \
    --model_name microsoft/mdeberta-v3-base \
    --dataset_name clef_1c \
    --language bg \
    --epochs 3 \
    --runs 1

# Step 3: Zero-shot evaluation in Bulgarian
!python eval_zero_cot.py \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --dataset_name clef_1c \
    --configuration zero_shot_generic \
    --task_labels ht \
    --label_type string \
    --language bg \
    --verbose

## 10. Running via Shell Script (SLURM Cluster)

If you're on a SLURM cluster, you can use `eval.sh` which generates
and submits a SLURM job script. Edit the configurable parameters at the
top of the file before running:

```bash
# Edit eval.sh to set:
#   DATASET_NAME, MODEL_NAME, TASK_LABELS, CONFIG, LANGUAGE, LABEL_TYPE
#
# Also set your HuggingFace token:
#   export HF_TOKEN="hf_your_token_here"

bash eval.sh
```

In [ ]:
# View the eval.sh script to see configurable parameters
!head -15 eval.sh